# Practice: AI 면접 질문 생성 에이전트 리팩토링

이 노트북은 `hk-llm-1day/project.ipynb`의 **AI 면접 질문 생성 에이전트**를 LangChain으로 다시 만들어보는 실습용 노트북입니다.

이번 실습의 목표는 완성형 리팩토링이 아니라 아래 3가지를 직접 손으로 익히는 것입니다.

- `ChatPromptTemplate`
- structured output (`PydanticOutputParser` 또는 `with_structured_output()`)
- `RunnableParallel`

핵심 로직은 일부러 `TODO`로 비워두었습니다. 직접 채우면서 학습해보세요.


## 섹션 1. 환경 준비

원본 노트북은 OpenAI SDK를 직접 호출하지만, 이번 실습에서는 LangChain 컴포넌트로 같은 흐름을 재구성합니다.

실습에서 주로 쓸 구성요소:

- `ChatOpenAI`
- `ChatPromptTemplate`
- `RunnableParallel`
- `RunnableLambda`
- `PydanticOutputParser` 또는 `with_structured_output()`


In [75]:
from pathlib import Path
from dotenv import load_dotenv

from pydantic import BaseModel, Field

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_openai import ChatOpenAI

from lib.phase2 import classify_personal_statement_sections

load_dotenv()


True

In [76]:
# TODO: ChatOpenAI 모델을 초기화하세요.
# 힌트:
# - model 이름을 명시하세요.
# - 처음에는 temperature=0 또는 낮은 값으로 시작해보세요.

model = ChatOpenAI(model="gpt-4o", temperature=0)


In [77]:
BASE_DIR = Path.cwd()
PROJECT_DIR = BASE_DIR if (BASE_DIR / "JD-sample.md").exists() else BASE_DIR / "hk-llm-1day"

jd_text = (PROJECT_DIR / "JD-sample.md").read_text(encoding="utf-8")
essay_text = (PROJECT_DIR / "essay-sample.md").read_text(encoding="utf-8")

print(PROJECT_DIR)
print(jd_text[:300])


c:\Users\user\working\langchain_v0_basics\hk-llm-1day\refactoring
# 삼성전자 DS부문 AI센터 SW개발 Job Description

---

## 📌 기본 정보

* **직무**: SW개발
* **근무지**: 경기도 수원, 기흥, 화성

AI 및 S/W 기술에 대한 전문 지식을 바탕으로,
DS부문 반도체 생산 및 공정의 지능화를 위한 AI 시스템을 개발하고,
S/W 개발 품질 향상 및 문화를 개선하는 직무

---

## 🧠 Role

* 반도체 도메인 특화 AI 모델, Agent 시스템/서비스, Platform 연구 및 개발
* 반도체 공정/수율/설비 데이터 기반 AI Agent 아키텍처 연


## 섹션 2. 단일 에이전트 체인 실습

원본 `Phase 1`은 `role`, `req`, `plus`, `biz`, `corp` 워커를 병렬 실행합니다. 하지만 학습할 때는 가장 작은 단위부터 시작하는 것이 좋습니다.

이 섹션에서는 `role` 워커 하나만 LangChain으로 옮겨봅니다.


In [78]:
def extract_section(markdown_text: str, section_name: str) -> str:
    """학습 편의를 위한 간단한 섹션 추출 헬퍼입니다."""
    marker = f"## {section_name}"
    start = markdown_text.find(marker)
    if start == -1:
        return ""

    remaining = markdown_text[start:]
    next_section_index = remaining.find("\n## ", len(marker))
    if next_section_index == -1:
        return remaining.strip()

    return remaining[:next_section_index].strip()


role_section = extract_section(jd_text, "🧠 Role")
print(role_section[:800])


## 🧠 Role

* 반도체 도메인 특화 AI 모델, Agent 시스템/서비스, Platform 연구 및 개발
* 반도체 공정/수율/설비 데이터 기반 AI Agent 아키텍처 연구 및 설계
* LLM 기반 추론 구조 및 멀티 Agent 협업 시스템 개발
* 반도체 분석 Workflow 자동화 Agent 설계 및 고도화
* Tool 연계, RAG 구조 및 도메인 지식 기반 추론 전략 구현
* Agent 성능, 추론 정확도, 응답 지연 및 비용 구조 최적화 연구
* 도메인 전문가 협업을 통한 AI 모델 및 시스템 고도화
* AI Agent 시스템의 실효 설계, 기술 검증(POC) 및 플랫폼 확장
* 반도체 설비/센서/운영 데이터 기반 이상 감지 및 예측 AI 모델 개발
* Autonomous Fab 구현을 위한 Physical AI 기반 Digital Twin 및 조치 자동화 기술 개발
* LLM/Agent 모델 Front-end 구현 및 운영
* 생산 일정, 자원 배치, 물류 운영 등 제조 전반의 최적화를 위한 AI 솔루션 개발
* AI 기반 의사 결정 지원 시스템 및 생산 계획 체계 구축
* 회로 설계 자동화 및 최적화 AI/ML 기술 개발
* AI Agent 서비스 플랫폼 및 AI 활용 Tool 개발 및 운영
* AI 모델 개발/Serving Platform 개발 및 운영

---


In [79]:
# TODO: role 분석 결과를 담을 Pydantic 모델을 정의하세요.
# 힌트:
# - 원본 phase1 결과처럼 roles 리스트를 가지게 해보세요.
# - 각 role은 role_name, required_skills, question_type 정도를 포함하면 좋습니다.

class RoleAnalysis(BaseModel):
    role_name: str = Field(..., description="직무 역할의 이름입니다.")
    required_skills: list[str] = Field(..., description="해당 역할에 필요한 기술 리스트입니다.")
    question_type: str = Field(..., description="이 역할에 대해 물어볼 질문의 유형입니다.")



In [80]:
# TODO: role 분석용 ChatPromptTemplate을 구성하세요.
# 힌트:
# - system 메시지: JD Role 섹션을 분석하는 전문가
# - human 메시지: {content}
# - 반환 형식은 roles 리스트 형태의 JSON/구조화 출력을 유도하세요.

role_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 JD의 Role 섹션을 분석하는 전문가입니다. 직무 섹션을 분석하여 각 역할에 대한 이름, 필요한 기술, 질문 유형을 추출하세요. 반환 형식은 roles 리스트 형태의 JSON/구조화 출력이어야 합니다. 반드시 한국어로 출력해야 합니다."),
        ("human", "{content}"),
    ]
)


In [81]:
# TODO: 아래 둘 중 하나를 선택해서 role_chain을 완성하세요.
# 1. PydanticOutputParser 사용
# 2. model.with_structured_output(RoleAnalysis) 사용

# 예시 흐름:
# role_chain = role_prompt | structured_model
# 또는
# role_chain = role_prompt | model | parser

role_chain = role_prompt | model.with_structured_output(RoleAnalysis)


In [82]:
# TODO: role_chain을 실행해보세요.
# result = role_chain.invoke({"content": role_section})
# result
result = role_chain.invoke({"content": role_section})
result

RoleAnalysis(role_name='반도체 도메인 특화 AI 연구 및 개발자', required_skills=['AI 모델 개발', 'Agent 시스템 설계', '플랫폼 연구 및 개발', '데이터 기반 AI 아키텍처 설계', 'LLM 기반 추론 구조 개발', '멀티 Agent 협업 시스템 개발', 'Workflow 자동화', 'Tool 연계 및 RAG 구조 구현', '도메인 지식 기반 추론 전략', '성능 최적화', '기술 검증(POC)', '이상 감지 및 예측 모델 개발', 'Digital Twin 기술', 'Front-end 구현 및 운영', 'AI 솔루션 개발', '의사 결정 지원 시스템 구축', '회로 설계 자동화 및 최적화', 'AI 서비스 플랫폼 개발'], question_type='기술적 문제 해결 및 시스템 설계 관련 질문')

## 섹션 3. Phase 1 다중 에이전트 확장

이제 `role` 하나에서 끝내지 않고, `req`, `plus`까지 확장해봅니다.

`biz`, `corp`는 원본 프로젝트에서 검색성 요소가 섞여 있으므로 이번 1차 실습에서는 제외합니다.


In [83]:
req_section = extract_section(jd_text, "✅ Requirements")
plus_section = extract_section(jd_text, "➕ Pluses")

print(req_section[:500])
print("\n---\n")
print(plus_section[:500])


## ✅ Requirements

* Python 등 프로그래밍 언어 기반 소프트웨어 개발 역량
* 자료구조 및 알고리즘 이해
* 머신러닝/딥러닝 모델 원리 이해 및 응용 역량
* 데이터 분석 및 통계 기반 문제 해결 능력
* 요구사항 기반 AI 시스템 설계/구현 역량
* Linux 환경 개발 및 모델 실행 경험
* Embedded 시스템 및 ARM Architecture 이해
* Windows/Linux 운영체제 이해
* 요구사항 기반 소프트웨어 설계 및 구현 능력

---

---




In [84]:
# TODO: RequirementAnalysis, PlusAnalysis 모델을 정의하세요.
# 힌트:
# - RequirementAnalysis: requirements 리스트
# - PlusAnalysis: pluses 리스트
# - 원본 phase1 결과의 필드를 참고해보세요.
class RequirementAnalysis(BaseModel):
    requirements: list[str] = Field(..., description="JD의 Requirements 섹션에서 추출된 요구사항 리스트입니다.")

class PlusAnalysis(BaseModel):
    pluses: list[str] = Field(..., description="JD의 Pluses 섹션에서 추출된 플러스 요인 리스트입니다.")


In [85]:
# TODO: req_prompt, plus_prompt를 각각 구성하세요.
# TODO: req_chain, plus_chain도 각각 완성하세요.

req_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 JD의 Requirements 섹션을 분석하는 전문가입니다. Requirements 섹션을 분석하여 요구사항 리스트를 추출하세요. 반환 형식은 requirements 리스트 형태의 JSON/구조화 출력이어야 합니다. 반드시 한국어로 출력해야 합니다."),
        ("human", "{content}"),
    ]
)

plus_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 JD의 Pluses 섹션을 분석하는 전문가입니다. Pluses 섹션을 분석하여 플러스 요인 리스트를 추출하세요. 반환 형식은 pluses 리스트 형태의 JSON/구조화 출력이어야 합니다. 반드시 한국어로 출력해야 합니다."),
        ("human", "{content}"),
    ]
)

req_chain = req_prompt | model.with_structured_output(RequirementAnalysis)
plus_chain = plus_prompt | model.with_structured_output(PlusAnalysis)


In [86]:
# TODO: RunnableParallel로 role/req/plus를 동시에 실행하세요.
# 힌트:
# - RunnableParallel은 기본적으로 같은 입력을 모든 체인에 넘깁니다.
# - 따라서 각 체인 앞에 RunnableLambda를 붙여 필요한 입력만 꺼내주면 편합니다.

parallel_chain = RunnableParallel(
    role=RunnableLambda(lambda x: x["role"]) | role_chain,
    req=RunnableLambda(lambda x: x["req"]) | req_chain,
    plus=RunnableLambda(lambda x: x["plus"]) | plus_chain,
)


In [87]:
# TODO: 병렬 실행 입력을 구성해보세요.
# 힌트: 각 체인이 서로 다른 섹션 문자열을 입력받도록 딕셔너리를 설계해야 합니다.

phase1_inputs = {
    "role": {"content": role_section},
    "req": {"content": req_section},
    "plus": {"content": plus_section},
}


In [88]:
# TODO: 병렬 결과를 보기 좋은 dict로 merge 해보세요.
# 힌트:
# - 이후 Phase 2가 그대로 참조할 수 있게 target 정보와 jd_analysis를 함께 두면 좋습니다.

def extract_job_title_from_jd(jd_text: str) -> str:
    for line in jd_text.splitlines():
        line = line.strip()
        if line.startswith("* **직무**:"):
            return line.split(":", 1)[1].strip()
    return "Unknown Job"


def extract_company_name_from_jd(jd_text: str) -> str:
    first_line = jd_text.splitlines()[0].strip() if jd_text.splitlines() else "Unknown Company"
    return first_line.replace("#", "").replace("Job Description", "").strip()


def merge_phase1_results(parallel_result):
    return {
        "role_analysis": parallel_result["role"],
        "requirement_analysis": parallel_result["req"],
        "plus_analysis": parallel_result["plus"],
    }


parallel_result = parallel_chain.invoke(phase1_inputs)

phase1_output = {
    "target": {
        "company": extract_company_name_from_jd(jd_text),
        "role": extract_job_title_from_jd(jd_text),
    },
    "jd_analysis": merge_phase1_results(parallel_result),
}

phase1_output


{'target': {'company': '삼성전자 DS부문 AI센터 SW개발', 'role': 'SW개발'},
 'jd_analysis': {'role_analysis': RoleAnalysis(role_name='반도체 도메인 특화 AI 연구 및 개발자', required_skills=['AI 모델 개발', 'Agent 시스템 설계', '플랫폼 연구 및 개발', '데이터 기반 AI 아키텍처 설계', 'LLM 기반 추론 구조 개발', '멀티 Agent 협업 시스템 개발', 'Workflow 자동화', 'Tool 연계 및 RAG 구조 구현', '도메인 지식 기반 추론 전략', '성능 최적화', '기술 검증(POC)', '이상 감지 및 예측 모델 개발', 'Digital Twin 기술', 'Front-end 구현', '제조 최적화 솔루션 개발', '의사 결정 지원 시스템 구축', '회로 설계 자동화 및 최적화', 'AI 서비스 플랫폼 운영'], question_type='기술적 문제 해결 및 시스템 설계 관련 질문'),
  'requirement_analysis': RequirementAnalysis(requirements=['Python 등 프로그래밍 언어 기반 소프트웨어 개발 역량', '자료구조 및 알고리즘 이해', '머신러닝/딥러닝 모델 원리 이해 및 응용 역량', '데이터 분석 및 통계 기반 문제 해결 능력', '요구사항 기반 AI 시스템 설계/구현 역량', 'Linux 환경 개발 및 모델 실행 경험', 'Embedded 시스템 및 ARM Architecture 이해', 'Windows/Linux 운영체제 이해', '요구사항 기반 소프트웨어 설계 및 구현 능력']),
  'plus_analysis': PlusAnalysis(pluses=['경쟁력 있는 급여 제공', '유연한 근무 시간', '재택 근무 옵션', '직원 복지 혜택', '전문성 개발 기회 제공', '긍정적인 근무 환경', '팀워크와 협력 강조', '리더십과 경영진의 지원', '다양성과 포용성 

## 섹션 4. Phase 2 문항 분석 실습

원본 `phase2.py`는 자소서를 문항별로 나누고, 각 문항을 JD와 매칭해 구조화된 결과를 만듭니다.

이번 실습에서는 먼저 문항을 분리한 뒤, **문항 1개를 분석하는 LangChain chain**을 만들어봅니다.


In [89]:
essay_items = classify_personal_statement_sections(essay_text)
len(essay_items), essay_items[0]


(4,
 {'question': '삼성전자를 지원한 이유와 입사 후 회사에서 이루고 싶은 꿈을 기술하십시오',
  'answer': '[40시간의 업무 병목을 해소한 설계 역량, 지연 없는 AI 서빙으로 이어가다]\n삼성전자 DS부문의 Autonomous Fab이 현장에 안착하려면 데이터를 지연 없이 처리하는 서빙 플랫폼의 성능이 보장되어야 합니다. 수많은 설비에서 발생하는 데이터를 실시간으로 분석하고 결과를 반환하는 과정에서 응답 지연이 발생하면 제조 공정 전체의 병목으로 이어집니다. 저는 삼성전자 네트워크사업부 연계 프로젝트에서 레거시 엑셀 중심으로 진행되던 업무를 웹 기반 협업 시스템으로 재설계해 40시간의 작업 시간을 1시간으로 단축한 경험이 있습니다. 현장 실무자의 업무 병목을 해결한 역량을 바탕으로 DS부문의 AI 플랫폼 안정성을 책임지기 위해 지원했습니다.\n\n입사 후 목표는 현장 엔지니어들이 지연 없이 사용할 수 있는 AI Agent 백엔드 환경을 구축하는 것입니다. 이를 위해 두 단계의 기술적 로드맵을 실행하겠습니다. 첫째, 반도체 공정에서 발생하는 방대한 비정형 데이터의 특성을 분석하고, 이에 최적화된 데이터베이스 I/O 통제 및 인덱스 설계 기준을 수립하겠습니다. 둘째, 무거운 AI 연산 결과가 대규모 트래픽 속에서도 안정적으로 반환될 수 있도록 캐싱 전략과 아키텍처를 단계적으로 고도화하겠습니다. 지속적인 트래픽 모니터링과 쿼리 튜닝을 통해 데이터 병목을 사전에 차단하고 견고한 실시간 서빙 인프라를 완성에 기여 하겠습니다.'})

In [90]:
# TODO: 자소서 문항 1개 분석 결과용 Pydantic 모델을 정의하세요.
# 힌트:
# - question_id
# - question_text
# - answer_text
# - item_type
# - question_intent
# - matched_jd
# - key_points
# - possible_risks

class EssayAnalysis(BaseModel):
    question_id: int = Field(..., description="문항의 고유 ID입니다.")
    question_text: str = Field(..., description="문항의 질문 텍스트입니다.")
    answer_text: str = Field(..., description="문항에 대한 지원자의 답변 텍스트입니다.")
    item_type: str = Field(..., description="문항의 유형입니다. 예: '자기소개', '경험기술', '기타'")
    question_intent: str = Field(..., description="문항의 의도를 분석한 텍스트입니다.")
    matched_jd: list[str] = Field(..., description="문항과 매칭되는 JD의 요소들입니다. 예: ['Role: Data Analyst', 'Requirement: Python 경험', 'Plus: AI 관련 경험']")
    key_points: list[str] = Field(..., description="문항에서 지원자가 반드시 언급해야 할 핵심 포인트입니다. 예: ['데이터 분석 경험', 'Python 활용 능력', 'AI 프로젝트 참여 경험']")
    possible_risks: list[str] = Field(..., description="지원자가 답변 작성 시 간과하기 쉬운 위험 요소들입니다. 예: ['경험이 부족한 기술을 과장해서 언급', '질문 의도에서 벗어난 답변 작성', '지원자의 진짜 강점이 드러나지 않음']")     


In [91]:
# TODO: 문항 1개를 분석하는 prompt를 작성하세요.
# 힌트:
# - 입력에는 JD 분석 결과와 statement_item이 모두 들어가야 합니다.
# - 원본 phase2.py의 system_prompt 역할을 참고해도 좋습니다.

essay_analysis_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 자소서 문항을 분석하는 전문가입니다. JD 분석 결과와 자소서 문항을 바탕으로, 문항의 의도, 지원자가 반드시 언급해야 할 핵심 포인트, 지원자가 간과하기 쉬운 위험 요소 등을 분석하세요. 반환 형식은 EssayAnalysis 모델 형태의 구조화 출력이어야 합니다. 반드시 한국어로 출력해야 합니다."
        ),
        (
            "human",
            """
[직무]
{job_category}

[JD 분석 결과]
{jd_summary}

[자소서 문항]
{question_text}

[답변]
{answer_text}

위 정보를 바탕으로 문항의 의도, JD와의 연결 요소, 핵심 포인트, 위험 요소를 분석해주세요.
"""
        )
    ]
)


In [92]:
# TODO: 문항 1개 분석 chain을 완성하세요.
# 힌트:
# - structured output을 붙이면 이후 문항 반복 적용이 훨씬 편해집니다.

essay_analysis_chain = essay_analysis_prompt | model.with_structured_output(EssayAnalysis)


In [93]:
# TODO: 우선 첫 번째 문항만 분석해보세요.
# 힌트:
# - Phase 2는 Phase 1의 jd_analysis를 입력으로 받아야 자연스럽게 이어집니다.

def format_phase1_for_prompt(phase1_output: dict) -> str:
    role_analysis = phase1_output["jd_analysis"]["role_analysis"]
    requirement_analysis = phase1_output["jd_analysis"]["requirement_analysis"]
    plus_analysis = phase1_output["jd_analysis"]["plus_analysis"]

    return f"""
[Role Analysis]
{role_analysis}

[Requirement Analysis]
{requirement_analysis}

[Plus Analysis]
{plus_analysis}
"""


statement_item = essay_items[0]

result = essay_analysis_chain.invoke(
    {
        "job_category": phase1_output["target"]["role"],
        "jd_summary": format_phase1_for_prompt(phase1_output),
        "question_text": statement_item["question"],
        "answer_text": statement_item["answer"],
    }
)
print(result)


question_id=1 question_text='삼성전자를 지원한 이유와 입사 후 회사에서 이루고 싶은 꿈을 기술하십시오' answer_text='[40시간의 업무 병목을 해소한 설계 역량, 지연 없는 AI 서빙으로 이어가다]\n삼성전자 DS부문의 Autonomous Fab이 현장에 안착하려면 데이터를 지연 없이 처리하는 서빙 플랫폼의 성능이 보장되어야 합니다. 수많은 설비에서 발생하는 데이터를 실시간으로 분석하고 결과를 반환하는 과정에서 응답 지연이 발생하면 제조 공정 전체의 병목으로 이어집니다. 저는 삼성전자 네트워크사업부 연계 프로젝트에서 레거시 엑셀 중심으로 진행되던 업무를 웹 기반 협업 시스템으로 재설계해 40시간의 작업 시간을 1시간으로 단축한 경험이 있습니다. 현장 실무자의 업무 병목을 해결한 역량을 바탕으로 DS부문의 AI 플랫폼 안정성을 책임지기 위해 지원했습니다.\n\n입사 후 목표는 현장 엔지니어들이 지연 없이 사용할 수 있는 AI Agent 백엔드 환경을 구축하는 것입니다. 이를 위해 두 단계의 기술적 로드맵을 실행하겠습니다. 첫째, 반도체 공정에서 발생하는 방대한 비정형 데이터의 특성을 분석하고, 이에 최적화된 데이터베이스 I/O 통제 및 인덱스 설계 기준을 수립하겠습니다. 둘째, 무거운 AI 연산 결과가 대규모 트래픽 속에서도 안정적으로 반환될 수 있도록 캐싱 전략과 아키텍처를 단계적으로 고도화하겠습니다. 지속적인 트래픽 모니터링과 쿼리 튜닝을 통해 데이터 병목을 사전에 차단하고 견고한 실시간 서빙 인프라를 완성에 기여 하겠습니다.' item_type='기타' question_intent='지원자가 삼성전자를 선택한 이유와 입사 후의 목표를 통해 회사와의 적합성을 평가하고, 지원자의 비전과 회사의 방향성이 일치하는지를 확인하고자 합니다.' matched_jd=['Role: 반도체 도메인 특화 AI 연구 및 개발자', 'Requirement: Python 등 프로그래밍 언어 기반 소프트웨어 개발 역량', 'Requirement: 데이

In [94]:
# TODO: 여러 문항에 대해 batch() 또는 반복 invoke를 적용해보세요.
# 힌트:
# - 처음에는 for loop가 더 이해하기 쉬운 방식입니다.
# - 그 다음 batch() 방식으로 바꿔보세요.

essay_inputs = [
    {
        "job_category": phase1_output["target"]["role"],
        "jd_summary": format_phase1_for_prompt(phase1_output),
        "question_text": item["question"],
        "answer_text": item["answer"],
    }
    for item in essay_items
]

results = essay_analysis_chain.batch(essay_inputs)
for r in results:
    print(r)


question_id=1 question_text='삼성전자를 지원한 이유와 입사 후 회사에서 이루고 싶은 꿈을 기술하십시오' answer_text='[40시간의 업무 병목을 해소한 설계 역량, 지연 없는 AI 서빙으로 이어가다]\n삼성전자 DS부문의 Autonomous Fab이 현장에 안착하려면 데이터를 지연 없이 처리하는 서빙 플랫폼의 성능이 보장되어야 합니다. 수많은 설비에서 발생하는 데이터를 실시간으로 분석하고 결과를 반환하는 과정에서 응답 지연이 발생하면 제조 공정 전체의 병목으로 이어집니다. 저는 삼성전자 네트워크사업부 연계 프로젝트에서 레거시 엑셀 중심으로 진행되던 업무를 웹 기반 협업 시스템으로 재설계해 40시간의 작업 시간을 1시간으로 단축한 경험이 있습니다. 현장 실무자의 업무 병목을 해결한 역량을 바탕으로 DS부문의 AI 플랫폼 안정성을 책임지기 위해 지원했습니다.\n\n입사 후 목표는 현장 엔지니어들이 지연 없이 사용할 수 있는 AI Agent 백엔드 환경을 구축하는 것입니다. 이를 위해 두 단계의 기술적 로드맵을 실행하겠습니다. 첫째, 반도체 공정에서 발생하는 방대한 비정형 데이터의 특성을 분석하고, 이에 최적화된 데이터베이스 I/O 통제 및 인덱스 설계 기준을 수립하겠습니다. 둘째, 무거운 AI 연산 결과가 대규모 트래픽 속에서도 안정적으로 반환될 수 있도록 캐싱 전략과 아키텍처를 단계적으로 고도화하겠습니다. 지속적인 트래픽 모니터링과 쿼리 튜닝을 통해 데이터 병목을 사전에 차단하고 견고한 실시간 서빙 인프라를 완성에 기여 하겠습니다.' item_type='기타' question_intent='지원자가 삼성전자를 선택한 이유와 입사 후의 목표를 통해 회사와의 적합성을 평가하고, 지원자의 비전과 목표가 회사의 방향성과 얼마나 일치하는지를 확인하고자 합니다.' matched_jd=['Role: 반도체 도메인 특화 AI 연구 및 개발자', 'Requirement: Python 등 프로그래밍 언어 기반 소프트웨어 개발 역량', 'Require

## 섹션 5. Context 생성 실습

원본 Phase 2의 후반부는 분석 결과를 기반으로 질문 생성용 `question_context`를 만드는 단계입니다.

이 단계는 LangChain에서 **이전 chain의 출력을 다음 chain의 입력으로 연결하는 연습**에 아주 좋습니다.


In [95]:
# TODO: question_context 결과 모델을 정의하세요.
# 힌트:
# - main_topics
# - verification_points
# - risk_points
# - followup_topics

class QuestionContext(BaseModel):
    main_topics: list[str] = Field(..., description="면접 질문을 만들 때 중심이 되는 핵심 주제 목록")
    verification_points: list[str] = Field(..., description="지원자의 주장이나 경험의 진위, 깊이, 구체성을 검증하기 위한 포인트 목록")
    risk_points: list[str] = Field(..., description="과장, 모호함, 논리 비약, 경험 부족 등 면접에서 추가 확인이 필요한 위험 요소 목록")
    followup_topics: list[str] = Field(..., description="꼬리질문으로 확장할 수 있는 후속 질문 주제 목록")        

QuestionContext


__main__.QuestionContext

In [96]:
# TODO: analysis 결과를 입력받아 question_context를 생성하는 prompt를 작성하세요.
question_context_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
당신은 시니어 면접관입니다.

주어진 자기소개서 분석 결과를 바탕으로,
실제 면접 질문을 설계하기 위한 질문 컨텍스트를 구조화하세요.

반드시 아래 기준을 반영하세요.
1. main_topics:
   - 면접에서 가장 먼저 확인해야 할 핵심 주제
   - JD 요구 역량과 자기소개서 경험이 연결되는 주제 중심

2. verification_points:
   - 지원자의 경험이 실제인지
   - 본인의 역할과 기여가 구체적인지
   - 기술 이해가 충분한지
   - 문제 해결 과정이 논리적인지
   를 검증할 수 있는 포인트

3. risk_points:
   - 과장 가능성
   - 경험 부족
   - 기술적 모호함
   - 직무와의 연결 부족
   - 답변의 일반론화 가능성
   같은 리스크를 정리

4. followup_topics:
   - 본 질문 이후 자연스럽게 이어질 수 있는 꼬리질문 주제

항목은 간결한 리스트 형태로 작성하세요.
중복은 피하고, 면접관 관점에서 실제로 쓸 수 있는 내용만 남기세요.
"""
        ),
        (
            "human",
            """
아래는 자기소개서 문항 분석 결과입니다.

question_id: {question_id}
question_text: {question_text}
answer_text: {answer_text}
item_type: {item_type}

question_intent:
{question_intent}

matched_jd:
{matched_jd}

key_points:
{key_points}

possible_risks:
{possible_risks}

이 분석 결과를 바탕으로 면접 질문 생성용 question_context를 만들어주세요.
"""
        ),
    ]
)



In [97]:
# TODO: context 생성 chain을 완성하세요.
# 힌트:
# - phase2_results는 이후 Phase 3에서 그대로 사용할 수 있도록
#   analyzed_item 정보 + question_context를 함께 담아두는 것이 좋습니다.

question_context_chain = question_context_prompt | model.with_structured_output(QuestionContext)

phase2_results = []

for analyzed_item in results:
    question_context = question_context_chain.invoke(
        {
            "question_id": analyzed_item.question_id,
            "question_text": analyzed_item.question_text,
            "answer_text": analyzed_item.answer_text,
            "item_type": analyzed_item.item_type,
            "question_intent": "\n".join(analyzed_item.question_intent) if isinstance(analyzed_item.question_intent, list) else str(analyzed_item.question_intent),
            "matched_jd": "\n".join(f"- {item}" for item in analyzed_item.matched_jd),
            "key_points": "\n".join(analyzed_item.key_points),
            "possible_risks": "\n".join(analyzed_item.possible_risks),
        }
    )

    merged_item = {
        "question_id": str(analyzed_item.question_id),
        "question_text": analyzed_item.question_text,
        "answer_text": analyzed_item.answer_text,
        "item_type": analyzed_item.item_type,
        "question_intent": analyzed_item.question_intent if isinstance(analyzed_item.question_intent, list) else [analyzed_item.question_intent],
        "matched_jd": analyzed_item.matched_jd,
        "key_points": analyzed_item.key_points,
        "possible_risks": analyzed_item.possible_risks,
        "question_context": question_context.model_dump(),
    }

    phase2_results.append(merged_item)

phase2_output = {
    "target": phase1_output["target"],
    "items": phase2_results,
}

phase2_output


{'target': {'company': '삼성전자 DS부문 AI센터 SW개발', 'role': 'SW개발'},
 'items': [{'question_id': '1',
   'question_text': '삼성전자를 지원한 이유와 입사 후 회사에서 이루고 싶은 꿈을 기술하십시오',
   'answer_text': '[40시간의 업무 병목을 해소한 설계 역량, 지연 없는 AI 서빙으로 이어가다]\n삼성전자 DS부문의 Autonomous Fab이 현장에 안착하려면 데이터를 지연 없이 처리하는 서빙 플랫폼의 성능이 보장되어야 합니다. 수많은 설비에서 발생하는 데이터를 실시간으로 분석하고 결과를 반환하는 과정에서 응답 지연이 발생하면 제조 공정 전체의 병목으로 이어집니다. 저는 삼성전자 네트워크사업부 연계 프로젝트에서 레거시 엑셀 중심으로 진행되던 업무를 웹 기반 협업 시스템으로 재설계해 40시간의 작업 시간을 1시간으로 단축한 경험이 있습니다. 현장 실무자의 업무 병목을 해결한 역량을 바탕으로 DS부문의 AI 플랫폼 안정성을 책임지기 위해 지원했습니다.\n\n입사 후 목표는 현장 엔지니어들이 지연 없이 사용할 수 있는 AI Agent 백엔드 환경을 구축하는 것입니다. 이를 위해 두 단계의 기술적 로드맵을 실행하겠습니다. 첫째, 반도체 공정에서 발생하는 방대한 비정형 데이터의 특성을 분석하고, 이에 최적화된 데이터베이스 I/O 통제 및 인덱스 설계 기준을 수립하겠습니다. 둘째, 무거운 AI 연산 결과가 대규모 트래픽 속에서도 안정적으로 반환될 수 있도록 캐싱 전략과 아키텍처를 단계적으로 고도화하겠습니다. 지속적인 트래픽 모니터링과 쿼리 튜닝을 통해 데이터 병목을 사전에 차단하고 견고한 실시간 서빙 인프라를 완성에 기여 하겠습니다.',
   'item_type': '기타',
   'question_intent': ['지원자가 삼성전자를 선택한 이유와 입사 후의 목표를 통해 회사와의 적합성을 평가하고, 지원자의 비전과 목표가 회사의 방향성과 얼마나 일치하는지를 확인하고

## 섹션 6. 중간 점검

여기까지는 원본 프로젝트의 **Phase 1 ~ Phase 2**를 LangChain으로 다시 구성한 구간입니다.

아래 질문에 답해보면서 지금까지 만든 구조를 한 번 점검해보세요.


In [ ]:
# TODO:
# 1. 원본 phase1.py와 비교했을 때 LangChain 버전의 장점은 무엇인가요?
# 2. 반대로 원본 코드가 더 직접적이라 이해가 쉬운 부분은 무엇인가요?
# 3. Phase 2 결과를 다음 단계(질문 생성)에 넘길 때 어떤 필드가 꼭 필요하다고 느꼈나요?


## 섹션 7. Phase 3 ~ Phase 5 확장

이제 첫 번째 노트북에서 만든 **phase2_output**을 그대로 이어 받아, 원본 프로젝트의 뒤쪽 흐름을 계속 구성합니다.

연결 흐름은 아래와 같습니다.

- phase2_output["target"]
- phase2_output["items"]
- Phase 3: 면접 질문 생성
- Phase 4: 질문 평가 및 개선 루프
- Phase 5: 질문 분류, 점수화, 랭킹, 리포트 출력


## 섹션 8. Phase 3 질문 생성 실습

Phase 3의 핵심은 Phase 2가 만든 `question_context`를 실제 면접 질문 세트로 바꾸는 것입니다.

원본 프로젝트 맥락에서는 단순히 질문을 만드는 것이 아니라 아래를 반영해야 합니다.

- JD 요구 역량
- 자소서 경험 근거
- 검증 포인트
- 리스크 포인트
- 꼬리질문으로 이어질 수 있는 주제


In [98]:
class GeneratedQuestion(BaseModel):
    question: str = Field(..., description="생성된 면접 질문 문장")
    question_type: str = Field(..., description="질문의 성격 예: 경험검증, 기술깊이, 의사결정")
    source_topic: str = Field(..., description="질문이 주로 검증하려는 원천 주제")
    intent: str = Field(..., description="면접관이 이 질문으로 확인하려는 핵심 의도")


class QuestionGenerationResult(BaseModel):
    question_id: str = Field(..., description="원본 자소서 문항 ID")
    generated_questions: list[GeneratedQuestion] = Field(..., description="생성된 면접 질문 목록")


In [99]:
# TODO: question generation prompt를 작성하세요.
# 힌트:
# - 질문은 4개를 생성하도록 요구해보세요.
# - 실제 경험 기반 답변 가능해야 한다는 조건을 넣으세요.
# - 질문 중복 금지, 꼬리질문 확장 가능성도 요구해보세요.

question_generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
당신은 시니어 면접관입니다.

주어진 자기소개서 분석 결과와 question_context를 바탕으로,
실제 실무 면접에서 사용할 수 있는 면접 질문 4개를 생성하세요.

반드시 아래 조건을 지키세요.
1. 질문은 지원자의 실제 경험을 바탕으로 답변할 수 있어야 합니다.
2. 질문은 JD 요구 역량과 연결되어야 합니다.
3. 질문은 구체적이어야 하며, 추상적이거나 뻔한 표현을 피해야 합니다.
4. 질문끼리는 서로 중복되지 않아야 합니다.
5. 각 질문은 꼬리질문으로 확장 가능한 형태여야 합니다.
6. key_points, verification_points, risk_points, followup_topics를 고르게 반영하세요.
7. 실제 면접관이 바로 사용할 수 있는 자연스러운 질문 문장으로 작성하세요.

질문 유형(question_type)은 예를 들어 아래 중 가장 적절한 것으로 정하세요.
- 경험검증
- 기술깊이
- 의사결정
- 직무적합
- 문제해결

source_topic에는 그 질문이 주로 검증하려는 핵심 주제를 적으세요.
intent에는 면접관이 이 질문으로 확인하려는 의도를 한 줄로 적으세요.
"""
        ),
        (
            "human",
            """
다음은 지원자의 자기소개서 분석 결과입니다.

[문항 ID]
{question_id}

[문항]
{question_text}

[답변]
{answer_text}

[item_type]
{item_type}

[JD 매칭]
{matched_jd}

[핵심 포인트]
{key_points}

[가능한 리스크]
{possible_risks}

[질문 컨텍스트 - main_topics]
{main_topics}

[질문 컨텍스트 - verification_points]
{verification_points}

[질문 컨텍스트 - risk_points]
{risk_points}

[질문 컨텍스트 - followup_topics]
{followup_topics}

위 정보를 바탕으로 면접 질문 4개를 생성하세요.
"""
        ),
    ]
)


In [100]:
# TODO: Phase 3 chain을 완성하세요.
# 힌트:
# - with_structured_output(QuestionGenerationResult)
# - prompt | structured_model

phase3_chain = question_generation_prompt | model.with_structured_output(QuestionGenerationResult)


In [101]:
def _get_value(item, key, default=None):
    if isinstance(item, dict):
        return item.get(key, default)
    return getattr(item, key, default)


def format_phase2_item_for_prompt(item) -> dict:
    raw_context = _get_value(item, "question_context", {}) or {}
    if isinstance(raw_context, dict):
        main_topics = raw_context.get("main_topics", [])
        verification_points = raw_context.get("verification_points", [])
        risk_points = raw_context.get("risk_points", [])
        followup_topics = raw_context.get("followup_topics", [])
    else:
        main_topics = getattr(raw_context, "main_topics", [])
        verification_points = getattr(raw_context, "verification_points", [])
        risk_points = getattr(raw_context, "risk_points", [])
        followup_topics = getattr(raw_context, "followup_topics", [])

    return {
        "question_id": str(_get_value(item, "question_id", "Q0")),
        "question_text": _get_value(item, "question_text", ""),
        "answer_text": _get_value(item, "answer_text", ""),
        "item_type": _get_value(item, "item_type", "unknown"),
        "matched_jd": "\n".join(f"- {x}" for x in (_get_value(item, "matched_jd", []) or [])),
        "key_points": "\n".join(f"- {x}" for x in (_get_value(item, "key_points", []) or [])),
        "possible_risks": "\n".join(f"- {x}" for x in (_get_value(item, "possible_risks", []) or [])),
        "main_topics": "\n".join(f"- {x}" for x in (main_topics or [])),
        "verification_points": "\n".join(f"- {x}" for x in (verification_points or [])),
        "risk_points": "\n".join(f"- {x}" for x in (risk_points or [])),
        "followup_topics": "\n".join(f"- {x}" for x in (followup_topics or [])),
    }


format_phase2_item_for_prompt(phase2_output["items"][0])


{'question_id': '1',
 'question_text': '삼성전자를 지원한 이유와 입사 후 회사에서 이루고 싶은 꿈을 기술하십시오',
 'answer_text': '[40시간의 업무 병목을 해소한 설계 역량, 지연 없는 AI 서빙으로 이어가다]\n삼성전자 DS부문의 Autonomous Fab이 현장에 안착하려면 데이터를 지연 없이 처리하는 서빙 플랫폼의 성능이 보장되어야 합니다. 수많은 설비에서 발생하는 데이터를 실시간으로 분석하고 결과를 반환하는 과정에서 응답 지연이 발생하면 제조 공정 전체의 병목으로 이어집니다. 저는 삼성전자 네트워크사업부 연계 프로젝트에서 레거시 엑셀 중심으로 진행되던 업무를 웹 기반 협업 시스템으로 재설계해 40시간의 작업 시간을 1시간으로 단축한 경험이 있습니다. 현장 실무자의 업무 병목을 해결한 역량을 바탕으로 DS부문의 AI 플랫폼 안정성을 책임지기 위해 지원했습니다.\n\n입사 후 목표는 현장 엔지니어들이 지연 없이 사용할 수 있는 AI Agent 백엔드 환경을 구축하는 것입니다. 이를 위해 두 단계의 기술적 로드맵을 실행하겠습니다. 첫째, 반도체 공정에서 발생하는 방대한 비정형 데이터의 특성을 분석하고, 이에 최적화된 데이터베이스 I/O 통제 및 인덱스 설계 기준을 수립하겠습니다. 둘째, 무거운 AI 연산 결과가 대규모 트래픽 속에서도 안정적으로 반환될 수 있도록 캐싱 전략과 아키텍처를 단계적으로 고도화하겠습니다. 지속적인 트래픽 모니터링과 쿼리 튜닝을 통해 데이터 병목을 사전에 차단하고 견고한 실시간 서빙 인프라를 완성에 기여 하겠습니다.',
 'item_type': '기타',
 'matched_jd': '- Role: 반도체 도메인 특화 AI 연구 및 개발자\n- Requirement: Python 등 프로그래밍 언어 기반 소프트웨어 개발 역량\n- Requirement: 데이터 분석 및 통계 기반 문제 해결 능력\n- Requirement: 요구사항 기반 AI 시스템 설계/구현 역량',
 'key_point

In [102]:
# TODO: 먼저 단일 문항에 대해 질문 4개를 생성해보세요.
# 힌트:
# - phase2_output["items"][0] 을 넣어보면 됩니다.

result = phase3_chain.invoke(format_phase2_item_for_prompt(phase2_output["items"][0]))
print(result)


question_id='1' generated_questions=[GeneratedQuestion(question='삼성전자 네트워크사업부 연계 프로젝트에서 40시간의 작업 시간을 1시간으로 단축했다고 하셨는데, 이 과정에서 구체적으로 어떤 역할을 하셨고, 어떤 기술적 도전이 있었는지 설명해 주실 수 있나요?', question_type='경험검증', source_topic='웹 기반 협업 시스템 재설계', intent='지원자가 프로젝트에서 맡았던 역할과 기술적 기여를 확인하기 위함입니다.'), GeneratedQuestion(question='AI 플랫폼의 안정성을 위해 제안하신 두 단계의 기술적 로드맵 중, 데이터베이스 I/O 통제 및 인덱스 설계 기준 수립에 대해 구체적으로 어떤 경험이 있으신가요?', question_type='기술깊이', source_topic='데이터베이스 I/O 통제 및 인덱스 설계', intent='지원자가 데이터베이스 설계와 관련된 구체적인 경험과 역량을 확인하기 위함입니다.'), GeneratedQuestion(question='입사 후 목표로 AI Agent 백엔드 환경 구축을 말씀하셨는데, 이를 실현하기 위해 예상되는 주요 도전 과제는 무엇이며, 어떻게 해결할 계획이신가요?', question_type='문제해결', source_topic='AI Agent 백엔드 환경 구축', intent='지원자가 목표를 실현하기 위한 구체적인 계획과 문제 해결 능력을 확인하기 위함입니다.'), GeneratedQuestion(question='반도체 공정에서 발생하는 비정형 데이터를 처리하기 위한 방법론에 대해 설명해 주실 수 있나요? 특히, 어떤 기술 스택을 활용할 계획이신가요?', question_type='기술깊이', source_topic='비정형 데이터 처리 방법', intent='지원자가 비정형 데이터 처리에 대한 이해와 기술적 접근 방식을 확인하기 위함입니다.')]


In [103]:
# TODO: 여러 문항으로 확장해보세요.
# 힌트:
# - 처음에는 for loop로 충분합니다.
# - 각 결과는 QuestionGenerationResult 형태로 phase3_results에 쌓아보세요.
# - 나중에 batch()나 RunnableLambda로 확장해보세요.

phase3_results = phase3_chain.batch([
    format_phase2_item_for_prompt(item)
    for item in phase2_output["items"]
])

for item in phase3_results:
    print(item)


question_id='1' generated_questions=[GeneratedQuestion(question='삼성전자 네트워크사업부 연계 프로젝트에서 40시간의 작업 시간을 1시간으로 단축했다고 하셨는데, 이 과정에서 구체적으로 어떤 역할을 하셨고, 어떤 기술적 도전이 있었는지 설명해 주실 수 있나요?', question_type='경험검증', source_topic='40시간의 작업 시간을 1시간으로 단축한 경험', intent='지원자가 프로젝트에서 맡았던 역할과 기여도를 확인하고, 기술적 도전과 해결 방법을 이해하기 위함입니다.'), GeneratedQuestion(question='웹 기반 협업 시스템으로 재설계할 때 사용한 기술 스택과 그 과정에서 직면한 주요 기술적 도전은 무엇이었나요?', question_type='기술깊이', source_topic='웹 기반 협업 시스템 재설계의 구체적인 기술 스택', intent='지원자가 사용한 기술 스택과 기술적 도전을 통해 문제 해결 능력을 평가하기 위함입니다.'), GeneratedQuestion(question='입사 후 목표로 제시한 AI Agent 백엔드 환경 구축 시, 예상되는 주요 도전 과제는 무엇이며, 이를 어떻게 해결할 계획인가요?', question_type='문제해결', source_topic='AI Agent 백엔드 환경 구축 시 예상되는 주요 도전 과제', intent='지원자가 목표를 실현하기 위해 예상되는 도전 과제를 어떻게 해결할 계획인지 평가하기 위함입니다.'), GeneratedQuestion(question='반도체 공정에서 발생하는 비정형 데이터를 처리하기 위한 데이터베이스 I/O 통제 및 인덱스 설계 기준을 수립한 경험이 있으신가요? 있다면 그 경험을 구체적으로 설명해 주세요.', question_type='경험검증', source_topic='데이터베이스 I/O 통제 및 인덱스 설계 기준 수립 경험', intent='지원자가 비정형 데이터 처리 경험을 통해 데이

In [104]:
# TODO: 생성 질문들을 하나의 리스트로 flatten 해보세요.
# 최종 목표는 Phase 4에 넣을 질문 문자열 리스트를 만드는 것입니다.
# 힌트:
# - phase3_results 안의 generated_questions를 순회하세요.
# - 우선은 question 문자열만 뽑아 phase4_questions에 담으면 됩니다.

phase4_questions = [
    q.question
    for item in phase3_results
    for q in item.generated_questions
]

phase4_questions



['삼성전자 네트워크사업부 연계 프로젝트에서 40시간의 작업 시간을 1시간으로 단축했다고 하셨는데, 이 과정에서 구체적으로 어떤 역할을 하셨고, 어떤 기술적 도전이 있었는지 설명해 주실 수 있나요?',
 '웹 기반 협업 시스템으로 재설계할 때 사용한 기술 스택과 그 과정에서 직면한 주요 기술적 도전은 무엇이었나요?',
 '입사 후 목표로 제시한 AI Agent 백엔드 환경 구축 시, 예상되는 주요 도전 과제는 무엇이며, 이를 어떻게 해결할 계획인가요?',
 '반도체 공정에서 발생하는 비정형 데이터를 처리하기 위한 데이터베이스 I/O 통제 및 인덱스 설계 기준을 수립한 경험이 있으신가요? 있다면 그 경험을 구체적으로 설명해 주세요.',
 '대학 시절 매일 정해진 시간보다 1시간 일찍 도착해 학습을 준비했다고 하셨는데, 이 습관이 실제로 어떤 학습 성과로 이어졌는지 구체적인 사례를 들어 설명해 주실 수 있나요?',
 'SSAFY 교육 과정에서 알고리즘 티어를 골드 1에서 플래티넘 4로 높였다고 하셨습니다. 이 과정에서 가장 어려웠던 알고리즘 문제는 무엇이었고, 이를 어떻게 해결했는지 설명해 주세요.',
 'SSAFY 교육에서 참여한 삼성전자 연계 프로젝트에 대해 설명해 주시고, 그 프로젝트에서 본인이 맡았던 역할과 기여한 부분은 무엇이었는지 구체적으로 말씀해 주세요.',
 '철학적 태도인 운명애와 자기 극복이 기술적 문제 해결에 어떻게 적용되었는지, 특히 AI 서빙 플랫폼 개발에서 이러한 태도가 어떤 영향을 미쳤는지 설명해 주실 수 있나요?',
 'CXL 기술을 도입한 경험이 있다고 하셨는데, 실제로 CXL을 활용하여 I/O 병목 문제를 해결한 프로젝트가 있었나요? 그 과정에서 직면한 주요 도전 과제와 이를 어떻게 극복했는지 설명해 주세요.',
 '애플리케이션 계층 최적화를 위해 인메모리 캐싱 전략과 쿼리 튜닝을 언급하셨습니다. 이전 프로젝트에서 이러한 최적화 기법을 적용한 구체적인 사례를 설명해 주시겠어요?',
 '반도체 제조 현장에서 데이터 처리 경험이 있다고

## 섹션 9. Phase 4 평가-최적화 루프 실습

이 단계는 원본 `lib/phase4.py`의 구조를 많이 참고하되, LangChain 체인과 Python 반복문을 함께 사용합니다.

핵심 평가 기준은 아래 5개입니다.

1. 꼬리질문으로 이어질 수 있는가
2. 질문이 구체적인가
3. JD 요구 역량과 연결되는가
4. 자기소개서 기반으로 답할 수 있는가
5. 실제 면접관이 할 법한 질문인가


In [105]:
class EvaluationResult(BaseModel):
    status: str = Field(..., description="PASS, FAIL, UNKNOWN 중 하나")
    feedback: str = Field(..., description="질문의 문제점과 개선 방향 요약")


class OptimizationAttempt(BaseModel):
    attempt: int = Field(..., description="몇 번째 개선 시도인지")
    optimized_question: str = Field(..., description="개선된 질문")
    optimized_status: str = Field(..., description="개선된 질문의 평가 상태")
    feedback_summary: str = Field(..., description="개선 이후 핵심 피드백 요약")


In [106]:
# TODO: evaluator prompt를 작성하세요.
# 힌트:
# - job_category와 question을 입력으로 받도록 하세요.
# - PASS/FAIL 판정을 명확하게 요구하세요.

evaluator_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
당신은 {job_category} 직무의 시니어 면접관입니다.

당신의 역할은 주어진 면접 질문이 실제 면접에서 사용할 만한 질문인지 평가하는 것입니다.

반드시 아래 기준으로 평가하세요.
1. 꼬리질문으로 자연스럽게 확장될 수 있는가
2. 질문이 구체적인가
3. 직무 요구 역량과 연결되는가
4. 지원자의 자기소개서/실제 경험을 바탕으로 답할 수 있는가
5. 실제 면접관이 할 법한 질문인가

판정 규칙:
- 위 기준을 전반적으로 충족하면 PASS
- 하나 이상 핵심적으로 부족하면 FAIL
- 애매하면 FAIL 쪽으로 판단하세요

feedback에는 아래 내용을 포함하세요.
- 왜 PASS 또는 FAIL인지
- 부족하다면 어떤 점이 문제인지
- 어떻게 개선하면 더 좋은 질문이 되는지

반드시 EvaluationResult 스키마에 맞게 응답하세요.
"""
        ),
        (
            "human",
            """
아래 면접 질문을 평가해주세요.

[직무]
{job_category}

[질문]
{question}
"""
        ),
    ]
)



In [107]:
# TODO: improver prompt를 작성하세요.
# 힌트:
# - 원본 질문과 직전 평가 결과(feedback)를 같이 넣어보세요.
# - 출력은 질문 1개만 나오도록 강하게 요구하세요.

improver_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
당신은 {job_category} 직무의 시니어 면접관입니다.

주어진 면접 질문은 이전 평가에서 FAIL 또는 보완 필요 판정을 받았습니다.
당신의 역할은 평가 피드백을 반영해, 더 좋은 면접 질문 1개로 개선하는 것입니다.

반드시 아래 조건을 지키세요.
1. 질문은 지원자의 실제 경험을 바탕으로 답할 수 있어야 합니다.
2. 질문은 구체적이어야 합니다.
3. 직무 요구 역량과 연결되어야 합니다.
4. 꼬리질문으로 확장 가능해야 합니다.
5. 실제 면접관이 바로 사용할 수 있는 자연스러운 질문이어야 합니다.
6. 원본 질문의 의도는 유지하되, 피드백에서 지적된 문제를 반드시 보완하세요.

출력 규칙:
- 질문은 반드시 1개만 생성하세요.
- 설명, 해설, 머리말 없이 개선된 질문만 반환하세요.
- 반드시 ImprovedQuestion 스키마에 맞게 응답하세요.
"""
        ),
        (
            "human",
            """
아래는 원본 질문과 직전 평가 결과입니다.

[직무]
{job_category}

[원본 질문]
{question}

[직전 평가 결과]
{feedback}

위 피드백을 반영하여 더 나은 면접 질문 1개로 개선해주세요.
"""
        ),
    ]
)


In [108]:
class ImprovedQuestion(BaseModel):
    optimized_question: str = Field(..., description="개선된 질문 1개")


In [109]:
# TODO: evaluator_chain, improver_chain을 각각 완성하세요.

evaluator_chain = evaluator_prompt | model.with_structured_output(EvaluationResult)
improver_chain = improver_prompt | model.with_structured_output(ImprovedQuestion)


In [110]:
def get_status(evaluation_result) -> str:
    """평가 결과 객체에서 PASS / FAIL / UNKNOWN만 추출하는 헬퍼"""
    if isinstance(evaluation_result, str):
        text = evaluation_result.upper()
        if "PASS" in text:
            return "PASS"
        if "FAIL" in text:
            return "FAIL"
        return "UNKNOWN"

    status = getattr(evaluation_result, "status", "UNKNOWN")
    return status


get_status("PASS")


'PASS'

In [111]:
# TODO: 질문 1개에 대한 retry 루프를 구현하세요.
# 힌트:
# - 최초 evaluator_chain 호출
# - FAIL이면 improver_chain 호출
# - 개선 질문을 다시 evaluator_chain으로 평가
# - max_retries 만큼 반복

def optimize_question_with_retries(question: str, job_category: str, max_retries: int = 3):
    original_question = question
    current_question = question
    attempts = []

    final_result = None
    final_question = current_question

    for i in range(max_retries):
        evaluation_result = evaluator_chain.invoke(
            {
                "question": current_question,
                "job_category": job_category,
            }
        )

        status = get_status(evaluation_result)

        if status == "PASS":
            final_result = evaluation_result
            final_question = current_question
            break

        improved_result = improver_chain.invoke(
            {
                "question": current_question,
                "job_category": job_category,
                "feedback": evaluation_result.feedback,
            }
        )

        improved_question = improved_result.optimized_question

        attempts.append(
            {
                "attempt": i + 1,
                "input_question": current_question,
                "evaluation_status": status,
                "evaluation_feedback": evaluation_result.feedback,
                "optimized_question": improved_question,
            }
        )

        current_question = improved_question
        final_question = current_question
        final_result = evaluation_result

    else:
        final_result = evaluator_chain.invoke(
            {
                "question": final_question,
                "job_category": job_category,
            }
        )

    return {
        "original_question": original_question,
        "attempts": attempts,
        "final_question": final_question,
        "final_status": get_status(final_result),
        "final_feedback": final_result.feedback,
    }


In [112]:
# TODO: 여러 질문에 대해 optimize_question_with_retries를 적용하세요.
# 힌트:
# - phase4_questions를 순회하면서 각 결과를 phase4_results에 쌓아보세요.
# - job_category는 Phase 1에서 JD로부터 추출한 값을 그대로 재사용하면 됩니다.

job_category = phase1_output["target"]["role"]

phase4_results = [
    optimize_question_with_retries(
        question=q,
        job_category=job_category,
        max_retries=3,
    )
    for q in phase4_questions
]

phase4_results


[{'original_question': '삼성전자 네트워크사업부 연계 프로젝트에서 40시간의 작업 시간을 1시간으로 단축했다고 하셨는데, 이 과정에서 구체적으로 어떤 역할을 하셨고, 어떤 기술적 도전이 있었는지 설명해 주실 수 있나요?',
  'attempts': [],
  'final_question': '삼성전자 네트워크사업부 연계 프로젝트에서 40시간의 작업 시간을 1시간으로 단축했다고 하셨는데, 이 과정에서 구체적으로 어떤 역할을 하셨고, 어떤 기술적 도전이 있었는지 설명해 주실 수 있나요?',
  'final_status': 'PASS',
  'final_feedback': '이 질문은 PASS입니다. \n\n1. 꼬리질문으로 자연스럽게 확장될 수 있는가: 지원자가 구체적으로 어떤 역할을 했는지, 어떤 기술적 도전이 있었는지를 설명하면서 다양한 꼬리질문으로 확장될 수 있습니다. 예를 들어, 사용한 기술, 문제 해결 과정, 팀과의 협업 방식 등에 대해 추가 질문이 가능합니다.\n\n2. 질문이 구체적인가: 질문은 매우 구체적입니다. 특정 프로젝트와 그 프로젝트에서의 성과를 언급하며, 지원자의 역할과 기술적 도전에 대해 묻고 있습니다.\n\n3. 직무 요구 역량과 연결되는가: SW개발 직무에서 중요한 문제 해결 능력, 기술적 역량, 프로젝트 관리 능력 등을 평가할 수 있는 질문입니다.\n\n4. 지원자의 자기소개서/실제 경험을 바탕으로 답할 수 있는가: 질문은 지원자가 자기소개서나 이력서에 기재했을 법한 구체적인 경험을 바탕으로 하고 있어, 지원자가 자신의 경험을 상세히 설명할 수 있습니다.\n\n5. 실제 면접관이 할 법한 질문인가: 면접관이 지원자의 구체적인 경험과 성과를 확인하기 위해 충분히 할 수 있는 질문입니다.'},
 {'original_question': '웹 기반 협업 시스템으로 재설계할 때 사용한 기술 스택과 그 과정에서 직면한 주요 기술적 도전은 무엇이었나요?',
  'attempts': [],
  'final_question': '웹

In [113]:
# TODO: PASS된 최종 질문만 추출하세요.
# 힌트:
# - phase4_results 안의 final_status를 확인하세요.
# - final_question만 모아서 phase4_pass_questions를 만들면 됩니다.

phase4_pass_questions = [
    item["final_question"]
    for item in phase4_results
    if item["final_status"] == "PASS"
]

phase4_pass_questions


['삼성전자 네트워크사업부 연계 프로젝트에서 40시간의 작업 시간을 1시간으로 단축했다고 하셨는데, 이 과정에서 구체적으로 어떤 역할을 하셨고, 어떤 기술적 도전이 있었는지 설명해 주실 수 있나요?',
 '웹 기반 협업 시스템으로 재설계할 때 사용한 기술 스택과 그 과정에서 직면한 주요 기술적 도전은 무엇이었나요?',
 '입사 후 목표로 제시한 AI Agent 백엔드 환경 구축 시, 예상되는 주요 도전 과제는 무엇이며, 이를 어떻게 해결할 계획인가요?',
 '반도체 공정에서 발생하는 비정형 데이터를 처리하기 위한 데이터베이스 I/O 통제 및 인덱스 설계 기준을 수립한 경험이 있으신가요? 있다면 그 경험을 구체적으로 설명해 주세요.',
 '대학 시절 매일 정해진 시간보다 1시간 일찍 도착해 학습을 준비했다고 하셨는데, 이 습관이 실제로 어떤 학습 성과로 이어졌는지 구체적인 사례를 들어 설명해 주실 수 있나요?',
 'SSAFY 교육 과정에서 알고리즘 티어를 골드 1에서 플래티넘 4로 높였다고 하셨습니다. 이 과정에서 가장 어려웠던 알고리즘 문제는 무엇이었고, 이를 어떻게 해결했는지 설명해 주세요.',
 'SSAFY 교육에서 참여한 삼성전자 연계 프로젝트에 대해 설명해 주시고, 그 프로젝트에서 본인이 맡았던 역할과 기여한 부분은 무엇이었는지 구체적으로 말씀해 주세요.',
 'AI 서빙 플랫폼 개발 프로젝트에서 직면했던 가장 큰 기술적 도전 과제는 무엇이었으며, 이를 해결하기 위해 어떤 접근 방식을 사용했는지 구체적으로 설명해 주실 수 있나요? 또한, 그 과정에서 배운 점이나 개선할 점이 있었다면 무엇인지 말씀해 주세요.',
 'CXL 기술을 도입한 경험이 있다고 하셨는데, 실제로 CXL을 활용하여 I/O 병목 문제를 해결한 프로젝트가 있었나요? 그 과정에서 직면한 주요 도전 과제와 이를 어떻게 극복했는지 설명해 주세요.',
 '애플리케이션 계층 최적화를 위해 인메모리 캐싱 전략과 쿼리 튜닝을 언급하셨습니다. 이전 프로젝트에서 이러한 최적화 기법을 적용한 구체적인 

## 섹션 10. Phase 5 질문 분류 및 랭킹 실습

이 단계에서는 PASS된 질문들을 최종 면접 준비용 리포트로 바꿉니다.

원본 `lib/phase5.py` 흐름을 다시 정리하면 아래와 같습니다.

1. 질문 유형 분류
2. 카테고리별 질문 점수화
3. 점수 기준 정렬
4. 카테고리별 랭킹 해설 생성
5. 최종 텍스트 리포트 출력


In [128]:
QUESTION_CATEGORIES = [
    "지원동기 / 직무 적합성",
    "프로젝트 경험 검증",
    "AI/LLM/Agent 역량 검증",
    "성능 최적화 / 시스템 설계",
    "인성 / 실행력 / 성장 가능성",
]

QUESTION_CATEGORIES


['지원동기 / 직무 적합성',
 '프로젝트 경험 검증',
 'AI/LLM/Agent 역량 검증',
 '성능 최적화 / 시스템 설계',
 '인성 / 실행력 / 성장 가능성']

In [129]:
class QuestionCategoryResult(BaseModel):
    category: str = Field(..., description="질문이 속하는 카테고리")


class QuestionScore(BaseModel):
    question: str = Field(..., description="질문 문장")
    difficulty: int = Field(..., description="난이도 점수")
    frequency: int = Field(..., description="빈출도 점수")
    relevance: int = Field(..., description="직무 연관도 점수")
    total: int = Field(..., description="총합 점수")
    reason: str = Field(..., description="점수 산정 이유")


class QuestionScoreList(BaseModel):
    scores: list[QuestionScore] = Field(..., description="질문 점수 목록")


In [130]:
# TODO: classifier prompt를 작성하세요.
# 힌트:
# - 5개 카테고리 중 반드시 하나만 고르게 하세요.
# - 출력 카테고리 이름은 원문 그대로 유지하게 하세요.

classifier_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
다음 면접 질문을 아래 5가지 유형 중 가장 적합한 하나로 분류하세요.

[유형]
- 지원동기 / 직무 적합성
- 프로젝트 경험 검증
- AI/LLM/Agent 역량 검증
- 성능 최적화 / 시스템 설계
- 인성 / 실행력 / 성장 가능성

[질문]
{question}

분류 결과는 반드시 위 [유형] 중 하나를 그대로 반환하세요.
다른 설명은 제외하세요.
반드시 한국어로 응답하세요.
반드시 QuestionCategoryResult 스키마에 맞게 응답하세요.
"""
        )
    ]
)

classifier_prompt


ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\n다음 면접 질문을 아래 5가지 유형 중 가장 적합한 하나로 분류하세요.\n\n[유형]\n- 지원동기 / 직무 적합성\n- 프로젝트 경험 검증\n- AI/LLM/Agent 역량 검증\n- 성능 최적화 / 시스템 설계\n- 인성 / 실행력 / 성장 가능성\n\n[질문]\n{question}\n\n분류 결과는 반드시 위 [유형] 중 하나를 그대로 반환하세요.\n다른 설명은 제외하세요.\n반드시 한국어로 응답하세요.\n반드시 QuestionCategoryResult 스키마에 맞게 응답하세요.\n'), additional_kwargs={})])

In [131]:
# TODO: scorer prompt를 작성하세요.
# 힌트:
# - 각 질문별로 난이도, 빈출도, 직무연관도를 평가하세요.
# - reason 필드는 반드시 한국어로 작성하게 하세요.

scorer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
당신은 {job_category} 직무 전문 면접관입니다.
아래 질문 목록에 대해 각각 난이도, 빈출도, 직무연관도를 평가하세요.(각 1~5점)
합계 점수(total)를 계산해주세요.

반환 형식:
```json
{{
  "scores": [
    {{
      "question": "질문 내용",
      "difficulty": 4,
      "frequency": 3,
      "relevance": 5,
      "total": 12,
      "reason": "점수 산정 근거 (간략히)"
    }}
  ]
}}
```

[질문 목록]
{questions}

JSON 외 출력 금지.
reason은 반드시 자연스러운 한국어로 작성하세요.
반드시 QuestionScoreList 스키마에 맞게 응답하세요.
"""
        )
    ]
)

scorer_prompt


ChatPromptTemplate(input_variables=['job_category', 'questions'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['job_category', 'questions'], input_types={}, partial_variables={}, template='\n당신은 {job_category} 직무 전문 면접관입니다.\n아래 질문 목록에 대해 각각 난이도, 빈출도, 직무연관도를 평가하세요.(각 1~5점)\n합계 점수(total)를 계산해주세요.\n\n반환 형식:\n```json\n{{\n  "scores": [\n    {{\n      "question": "질문 내용",\n      "difficulty": 4,\n      "frequency": 3,\n      "relevance": 5,\n      "total": 12,\n      "reason": "점수 산정 근거 (간략히)"\n    }}\n  ]\n}}\n```\n\n[질문 목록]\n{questions}\n\nJSON 외 출력 금지.\nreason은 반드시 자연스러운 한국어로 작성하세요.\n반드시 QuestionScoreList 스키마에 맞게 응답하세요.\n'), additional_kwargs={})])

In [132]:
# TODO: ranking reason prompt를 작성하세요.
# 힌트:
# - 카테고리별 준비 포인트를 3~4문장으로 요약하게 하세요.
# - summary는 반드시 한국어로 출력하게 하세요.

ranking_reason_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
당신은 {job_category} 직무 면접관입니다.
다음은 [{category}] 유형의 질문들을 중요도(합계 점수) 순으로 정렬한 것입니다.

[순위별 질문]
{ranked_questions}

지원자가 이 유형의 질문들에 대비하기 위한 종합적인 핵심 포인트(순위 해설, 합격 팁 등)를 3~4문장으로 요약해주세요.
summary는 반드시 자연스러운 한국어 문장으로만 작성하세요.
영어 단어 나열이나 번역투 표현을 피하고, 실제 면접 준비 가이드처럼 써주세요.
반드시 RankingReason 스키마에 맞게 응답하세요.
"""
        )
    ]
)

ranking_reason_prompt


ChatPromptTemplate(input_variables=['category', 'job_category', 'ranked_questions'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['category', 'job_category', 'ranked_questions'], input_types={}, partial_variables={}, template='\n당신은 {job_category} 직무 면접관입니다.\n다음은 [{category}] 유형의 질문들을 중요도(합계 점수) 순으로 정렬한 것입니다.\n\n[순위별 질문]\n{ranked_questions}\n\n지원자가 이 유형의 질문들에 대비하기 위한 종합적인 핵심 포인트(순위 해설, 합격 팁 등)를 3~4문장으로 요약해주세요.\nsummary는 반드시 자연스러운 한국어 문장으로만 작성하세요.\n영어 단어 나열이나 번역투 표현을 피하고, 실제 면접 준비 가이드처럼 써주세요.\n반드시 RankingReason 스키마에 맞게 응답하세요.\n'), additional_kwargs={})])

In [133]:
class RankingReason(BaseModel):
    summary: str = Field(..., description="카테고리별 준비 포인트 요약")


In [134]:
# Phase 5용 chain들을 구성합니다.

classifier_chain = classifier_prompt | model.with_structured_output(QuestionCategoryResult)
scorer_chain = scorer_prompt | model.with_structured_output(QuestionScoreList)
ranking_reason_chain = ranking_reason_prompt | model.with_structured_output(RankingReason)

classifier_chain, scorer_chain, ranking_reason_chain


(ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\n다음 면접 질문을 아래 5가지 유형 중 가장 적합한 하나로 분류하세요.\n\n[유형]\n- 지원동기 / 직무 적합성\n- 프로젝트 경험 검증\n- AI/LLM/Agent 역량 검증\n- 성능 최적화 / 시스템 설계\n- 인성 / 실행력 / 성장 가능성\n\n[질문]\n{question}\n\n분류 결과는 반드시 위 [유형] 중 하나를 그대로 반환하세요.\n다른 설명은 제외하세요.\n반드시 한국어로 응답하세요.\n반드시 QuestionCategoryResult 스키마에 맞게 응답하세요.\n'), additional_kwargs={})])
 | RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001890B724A90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001890BFC7510>, root_client=<openai.OpenAI object at 0x000001890B4BAD50>, root_async_client=<openai.AsyncOpenAI object at 0x000001890C38BC10>, model_name='gpt-4o', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream

In [135]:
# PASS된 질문들을 카테고리별로 묶어봅니다.

def group_questions_by_category(questions: list[str], job_category: str):
    grouped = {category: [] for category in QUESTION_CATEGORIES}

    for question in questions:
        category_result = classifier_chain.invoke(
            {
                "job_category": job_category,
                "question": question,
            }
        )

        category = category_result.category
        if category not in grouped:
            grouped[category] = []
        grouped[category].append(question)

    return {k: v for k, v in grouped.items() if v}


group_questions_by_category(phase4_pass_questions[:2], job_category) if phase4_pass_questions else {}


{'프로젝트 경험 검증': ['삼성전자 네트워크사업부 연계 프로젝트에서 40시간의 작업 시간을 1시간으로 단축했다고 하셨는데, 이 과정에서 구체적으로 어떤 역할을 하셨고, 어떤 기술적 도전이 있었는지 설명해 주실 수 있나요?',
  '웹 기반 협업 시스템으로 재설계할 때 사용한 기술 스택과 그 과정에서 직면한 주요 기술적 도전은 무엇이었나요?']}

In [136]:
# 분류 -> 점수화 -> 정렬 -> 카테고리 요약 생성을 수행합니다.

grouped_questions = group_questions_by_category(phase4_pass_questions, job_category)

phase5_result = {}

for category, questions in grouped_questions.items():
    questions_text = "\n".join(f"- {q}" for q in questions)

    score_result = scorer_chain.invoke(
        {
            "job_category": job_category,
            "category": category,
            "questions": questions_text,
        }
    )

    ranked_questions = sorted(
        [item.model_dump() for item in score_result.scores],
        key=lambda x: x["total"],
        reverse=True,
    )

    ranked_text = "\n".join(
        f"{idx + 1}. {item['question']} (총점: {item['total']})"
        for idx, item in enumerate(ranked_questions)
    )

    ranking_reason = ranking_reason_chain.invoke(
        {
            "job_category": job_category,
            "category": category,
            "ranked_questions": ranked_text,
        }
    )

    phase5_result[category] = {
        "question_count": len(ranked_questions),
        "ranked_questions": ranked_questions,
        "ranking_reason": ranking_reason.summary,
    }

phase5_result


{'지원동기 / 직무 적합성': {'question_count': 1,
  'ranked_questions': [{'question': '삼성전자 제품 사용 경험이 지원 직무와 어떻게 연결된다고 생각하시나요? 구체적인 예시를 들어 설명해 주세요.',
    'difficulty': 4,
    'frequency': 3,
    'relevance': 5,
    'total': 12,
    'reason': '이 질문은 지원자가 삼성전자 제품에 대한 이해도를 바탕으로 직무와의 연관성을 설명할 수 있는지를 평가하는 데 유용합니다. 따라서 직무 연관도가 높습니다. 그러나 구체적인 예시를 요구하기 때문에 난이도가 다소 높을 수 있습니다. 빈출도는 중간 정도로, 특정 기업 면접에서 자주 나올 수 있는 질문입니다.'}],
  'ranking_reason': '삼성전자 제품 사용 경험을 직무와 연결짓는 질문은 지원자의 실질적인 경험과 직무 이해도를 평가하는 데 중요합니다. 지원자는 제품 사용 경험을 통해 얻은 인사이트가 직무 수행에 어떻게 기여할 수 있는지를 구체적으로 설명해야 합니다. 예를 들어, 특정 제품의 기능이나 문제점을 개선할 수 있는 아이디어를 제시하거나, 제품 사용 경험을 통해 얻은 기술적 지식을 직무에 적용할 수 있는 방안을 제시하는 것이 좋습니다.'},
 '프로젝트 경험 검증': {'question_count': 9,
  'ranked_questions': [{'question': '삼성전자 네트워크사업부 연계 프로젝트에서 40시간의 작업 시간을 1시간으로 단축했다고 하셨는데, 이 과정에서 구체적으로 어떤 역할을 하셨고, 어떤 기술적 도전이 있었는지 설명해 주실 수 있나요?',
    'difficulty': 4,
    'frequency': 3,
    'relevance': 5,
    'total': 12,
    'reason': '이 질문은 프로젝트 관리 및 기술적 문제 해결 능력을 평가하는 데 중요합니다. 삼성전자와의 연계 프로젝트라는 점

In [137]:
# 최종 리포트 출력 함수를 작성합니다.

def format_phase5_output(result: dict) -> str:
    if not result:
        return "출력할 Phase 5 결과가 없습니다."

    lines = [f"{job_category} 직무 면접 질문 준비 리포트", ""]

    ordered_categories = [category for category in QUESTION_CATEGORIES if category in result]
    ordered_categories += [category for category in result if category not in ordered_categories]

    for category in ordered_categories:
        data = result[category]
        lines.append(f"[카테고리] {category}")
        lines.append(f"질문 수: {data['question_count']}")

        for idx, item in enumerate(data["ranked_questions"], start=1):
            lines.append(
                f"{idx}. {item['question']} (난이도={item['difficulty']}, 빈출도={item['frequency']}, 직무연관도={item['relevance']}, 총점={item['total']})"
            )
            lines.append(f"   이유: {item['reason']}")

        lines.append(f"요약: {data['ranking_reason']}")
        lines.append("")

    return "\n".join(lines).strip()


print(format_phase5_output(phase5_result) if phase5_result else "phase5_result를 먼저 만들어주세요.")


SW개발 직무 면접 질문 준비 리포트

[카테고리] 지원동기 / 직무 적합성
질문 수: 1
1. 삼성전자 제품 사용 경험이 지원 직무와 어떻게 연결된다고 생각하시나요? 구체적인 예시를 들어 설명해 주세요. (난이도=4, 빈출도=3, 직무연관도=5, 총점=12)
   이유: 이 질문은 지원자가 삼성전자 제품에 대한 이해도를 바탕으로 직무와의 연관성을 설명할 수 있는지를 평가하는 데 유용합니다. 따라서 직무 연관도가 높습니다. 그러나 구체적인 예시를 요구하기 때문에 난이도가 다소 높을 수 있습니다. 빈출도는 중간 정도로, 특정 기업 면접에서 자주 나올 수 있는 질문입니다.
요약: 삼성전자 제품 사용 경험을 직무와 연결짓는 질문은 지원자의 실질적인 경험과 직무 이해도를 평가하는 데 중요합니다. 지원자는 제품 사용 경험을 통해 얻은 인사이트가 직무 수행에 어떻게 기여할 수 있는지를 구체적으로 설명해야 합니다. 예를 들어, 특정 제품의 기능이나 문제점을 개선할 수 있는 아이디어를 제시하거나, 제품 사용 경험을 통해 얻은 기술적 지식을 직무에 적용할 수 있는 방안을 제시하는 것이 좋습니다.

[카테고리] 프로젝트 경험 검증
질문 수: 9
1. 삼성전자 네트워크사업부 연계 프로젝트에서 40시간의 작업 시간을 1시간으로 단축했다고 하셨는데, 이 과정에서 구체적으로 어떤 역할을 하셨고, 어떤 기술적 도전이 있었는지 설명해 주실 수 있나요? (난이도=4, 빈출도=3, 직무연관도=5, 총점=12)
   이유: 이 질문은 프로젝트 관리 및 기술적 문제 해결 능력을 평가하는 데 중요합니다. 삼성전자와의 연계 프로젝트라는 점에서 직무 연관도가 높습니다.
2. 웹 기반 협업 시스템으로 재설계할 때 사용한 기술 스택과 그 과정에서 직면한 주요 기술적 도전은 무엇이었나요? (난이도=3, 빈출도=4, 직무연관도=5, 총점=12)
   이유: 웹 기반 시스템 설계는 SW 개발에서 빈번히 발생하는 과제입니다. 기술 스택 선택과 문제 해결 능력을 평가할 수 있어 직무 연관도가 높습니다.
3. 반도

## 섹션 11. End-to-End 연결

이제 최종 목표는 아래 흐름을 실제로 이어보는 것입니다.

- Phase 2 결과 1개 이상 준비
- Phase 3 질문 생성
- Phase 4 평가 및 개선
- PASS 질문 추출
- Phase 5 분류 및 리포트 생성

처음에는 문항 1개만으로 연결해도 충분합니다.


In [138]:
# 최종 흐름이 잘 이어졌는지 확인합니다.

print("=== target 정보 ===")
print(phase2_output["target"])
print()

print("=== Phase 3 질문 수 ===")
print(len(phase4_questions))
print()

print("=== Phase 4 PASS 질문 수 ===")
print(len(phase4_pass_questions))
print()

print("=== Phase 5 최종 리포트 ===")
print(format_phase5_output(phase5_result))


=== target 정보 ===
{'company': '삼성전자 DS부문 AI센터 SW개발', 'role': 'SW개발'}

=== Phase 3 질문 수 ===
16

=== Phase 4 PASS 질문 수 ===
16

=== Phase 5 최종 리포트 ===
SW개발 직무 면접 질문 준비 리포트

[카테고리] 지원동기 / 직무 적합성
질문 수: 1
1. 삼성전자 제품 사용 경험이 지원 직무와 어떻게 연결된다고 생각하시나요? 구체적인 예시를 들어 설명해 주세요. (난이도=4, 빈출도=3, 직무연관도=5, 총점=12)
   이유: 이 질문은 지원자가 삼성전자 제품에 대한 이해도를 바탕으로 직무와의 연관성을 설명할 수 있는지를 평가하는 데 유용합니다. 따라서 직무 연관도가 높습니다. 그러나 구체적인 예시를 요구하기 때문에 난이도가 다소 높을 수 있습니다. 빈출도는 중간 정도로, 특정 기업 면접에서 자주 나올 수 있는 질문입니다.
요약: 삼성전자 제품 사용 경험을 직무와 연결짓는 질문은 지원자의 실질적인 경험과 직무 이해도를 평가하는 데 중요합니다. 지원자는 제품 사용 경험을 통해 얻은 인사이트가 직무 수행에 어떻게 기여할 수 있는지를 구체적으로 설명해야 합니다. 예를 들어, 특정 제품의 기능이나 문제점을 개선할 수 있는 아이디어를 제시하거나, 제품 사용 경험을 통해 얻은 기술적 지식을 직무에 적용할 수 있는 방안을 제시하는 것이 좋습니다.

[카테고리] 프로젝트 경험 검증
질문 수: 9
1. 삼성전자 네트워크사업부 연계 프로젝트에서 40시간의 작업 시간을 1시간으로 단축했다고 하셨는데, 이 과정에서 구체적으로 어떤 역할을 하셨고, 어떤 기술적 도전이 있었는지 설명해 주실 수 있나요? (난이도=4, 빈출도=3, 직무연관도=5, 총점=12)
   이유: 이 질문은 프로젝트 관리 및 기술적 문제 해결 능력을 평가하는 데 중요합니다. 삼성전자와의 연계 프로젝트라는 점에서 직무 연관도가 높습니다.
2. 웹 기반 협업 시스템으로 재설계할 때 사용한 기술 스택과 그 과정에